# 03 — Define Material, Load, and Constraints

This notebook records the analysis configuration in an explicit JSON file and prints the
material, load, boundary condition, and verification criteria used by the pipeline.

## What this notebook does

- Defines the reusable analysis configuration object.
- Documents the mm–N–MPa unit convention used in the CalculiX input deck.
- Saves the configuration to `analysis_config.json`.

In [1]:
import json
import logging
import sys
from dataclasses import asdict, is_dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

logging.basicConfig(level=logging.DEBUG, format='%(levelname)s %(name)s: %(message)s')

def find_module_root(start: Path | None = None) -> Path:
    current = start or Path.cwd()
    for candidate in [current, *current.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate the fea_cad_one_sample module root.')

def json_safe(value: Any) -> Any:
    if is_dataclass(value):
        return json_safe(asdict(value))
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]
    return value

def write_json(path: Path, payload: Any) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True) + '\n', encoding='utf-8')
    return path

def load_json(path: Path) -> dict[str, Any]:
    return json.loads(Path(path).read_text(encoding='utf-8'))

MODULE_ROOT = find_module_root()
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

import src.interfaces as iface

plt.style.use('seaborn-v0_8-whitegrid')
print(f'[SETUP] MODULE_ROOT={MODULE_ROOT}')

DEBUG matplotlib.pyplot: Loaded backend Agg version v2.2.
DEBUG trimesh.util: searching for blender in: /Users/vedaangchopra/.local/bin:/bin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/bin:/Users/vedaangchopra/.nvm/versions/node/v25.6.0/bin:/opt/homebrew/Caskroom/miniconda/base/envs/cad_physics/bin:/Users/vedaangchopra/.lmstudio/bin:/Users/vedaangchopra/.cargo/bin:/opt/homebrew/bin:/Applications/blender.app/Contents/MacOS:/usr/bin:/usr/local/bin:/pkg/env/global/bin:/Applications/Blender/blender.app/Contents/MacOS:/Users/vedaangchopra/.antigravity/antigravity/bin:/Library/TeX/texbin:/sbin:/opt/homebrew/Caskroom/miniconda/base/condabin:/opt/homebrew/sbin:/Library/Apple/usr/bin:/Applications/Blender.app/Contents/MacOS:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/appleinternal/bin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/local/bin:/opt/homebrew/opt/openjdk@17/bin:/usr/sbin:/System/Cryptexes/App/usr/bin:/Users/vedaangchopra/.bun/

[SETUP] MODULE_ROOT=/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample


In [2]:
RUN_DIR = MODULE_ROOT / 'outputs' / 'sample_00689964' / '01_dataset_original'
SOURCE_STEP_PATH = RUN_DIR / "original.step" 
MESH_SIZE_MM = 8.0
LOAD_MAGNITUDE_N = 100.0

print('[STAGE] configuration: build and save analysis config')
config = iface.build_baseline_config(
    run_dir=RUN_DIR,
    source_step_path=SOURCE_STEP_PATH,
    mesh_size_mm=MESH_SIZE_MM,
    load_magnitude_n=LOAD_MAGNITUDE_N,
)
analysis_config_path = RUN_DIR / 'analysis_config.json'
write_json(analysis_config_path, config)

print('  analysis_config_path =', analysis_config_path)
print('  material =', json.dumps(json_safe(config.material), indent=2))
print('  load =', json.dumps(json_safe(config.load), indent=2))
print('  boundary_condition =', json.dumps(json_safe(config.boundary_condition), indent=2))
print('  verification_criteria =', json.dumps(json_safe(config.verification_criteria), indent=2))
print('  solver_executable =', config.solver_executable)
print('  units = mm, N, MPa in the deck; config stores SI values in Pa and converts on write')
df = pd.DataFrame([
    {'field': 'material.name', 'value': config.material.name},
    {'field': 'material.youngs_modulus_pa', 'value': config.material.youngs_modulus_pa},
    {'field': 'material.poissons_ratio', 'value': config.material.poissons_ratio},
    {'field': 'material.yield_strength_pa', 'value': config.material.yield_strength_pa},
    {'field': 'load.magnitude_n', 'value': config.load.magnitude_n},
    {'field': 'load.direction_vector', 'value': config.load.direction_vector},
    {'field': 'load.target_node_set', 'value': config.load.target_node_set},
    {'field': 'boundary_condition.fixed_node_set', 'value': config.boundary_condition.fixed_node_set},
    {'field': 'verification.max_displacement_mm', 'value': config.verification_criteria.max_displacement_mm},
    {'field': 'verification.required_safety_factor', 'value': config.verification_criteria.required_safety_factor},
])
display(df)
print('✓ configuration recorded')

INFO src.fea_replication.pipeline: build_baseline_config | start | run_dir=/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/01_dataset_original | source_step_path=/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/01_dataset_original/original.step | mesh_size_mm=8.0 | load_magnitude_n=100.0
INFO src.fea_replication.pipeline: build_baseline_config | done | run_dir=/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/01_dataset_original


[STAGE] configuration: build and save analysis config
  analysis_config_path = /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/01_dataset_original/analysis_config.json
  material = {
  "name": "Aluminum 6061",
  "youngs_modulus_pa": 69000000000.0,
  "poissons_ratio": 0.33,
  "yield_strength_pa": 276000000.0
}
  load = {
  "magnitude_n": 100.0,
  "direction_vector": [
    0.0,
    0.0,
    -1.0
  ],
  "target_region": "free end face near maximum axis",
  "target_node_set": "LOAD_END"
}
  boundary_condition = {
  "fixed_region": "fixed end face near min axis",
  "fixed_node_set": "FIXED_END"
}
  verification_criteria = {
  "max_displacement_mm": 2.0,
  "required_safety_factor": 2.0
}
  solver_executable = ccx
  units = mm, N, MPa in the deck; config stores SI values in Pa and converts on write


,field,value
0,material.name,Aluminum 6061
1,material.youngs_modulus_pa,69000000000.0
2,material.poissons_ratio,0.33
3,material.yield_strength_pa,276000000.0
4,load.magnitude_n,100.0
5,load.direction_vector,"(0.0, 0.0, -1.0)"
6,load.target_node_set,LOAD_END
7,boundary_condition.fixed_node_set,FIXED_END
8,verification.max_displacement_mm,2.0
9,verification.required_safety_factor,2.0


✓ configuration recorded
